# Telecom OSS Anomaly Detection - Member 1

Scope: load data, perform EDA, preprocess, select features, train Isolation Forest, compute anomaly scores, and save downstream artifacts.

## 1. Imports

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from anomaly_detector import (
    compute_anomaly_scores,
    evaluation_summary,
    predict_anomalies,
    save_anomaly_scores,
    save_model,
    scores_to_dataframe,
    to_json_records,
    train_model,
)
from config import (
    FEATURE_REPORT_PATH,
    MODEL_PATH,
    PREPROCESSING_PIPELINE_PATH,
)
from data_loader import (
    load_data,
    sample_data,
    save_processed_data,
    train_test_split_data,
)
from feature_selection import select_features
from preprocessing import (
    build_preprocessing_pipeline,
    clean_data,
    detect_outliers,
    handle_missing_values,
)

sns.set_theme(style="whitegrid")

## 2. Dataset Loading

In [ ]:
raw_data = load_data()
raw_data.head()

## 3. EDA

In [ ]:
print(f"Dataset shape: {raw_data.shape}")
display(raw_data.info())
display(raw_data.describe(include="all").T)
display(raw_data.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
numeric_columns = raw_data.select_dtypes(include="number").columns

plt.figure(figsize=(12, 8))
sns.heatmap(raw_data[numeric_columns].corr(), cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Sampling

In [ ]:
sampled_data = sample_data(raw_data)
print(f"Sampled shape: {sampled_data.shape}")
sampled_data.head()

## 5. Train/Test Split

In [ ]:
train_data, test_data = train_test_split_data(sampled_data)
print(f"Train shape: {train_data.shape}")
print(f"Test shape: {test_data.shape}")

## 6. Preprocessing

In [ ]:
train_clean = handle_missing_values(clean_data(train_data))
test_clean = handle_missing_values(clean_data(test_data))

train_with_outlier_flags = detect_outliers(train_clean)
display(train_with_outlier_flags["eda_outlier_flag"].value_counts())

preprocessing_pipeline, raw_feature_columns = build_preprocessing_pipeline(train_clean)
train_processed_array = preprocessing_pipeline.fit_transform(train_clean)
test_processed_array = preprocessing_pipeline.transform(test_clean)

processed_feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out()
train_processed = pd.DataFrame(train_processed_array, columns=processed_feature_names)
test_processed = pd.DataFrame(test_processed_array, columns=processed_feature_names)

save_processed_data(train_processed, test_processed)
train_processed.head()

## 7. Feature Selection

In [ ]:
train_selected, selected_features, feature_report = select_features(train_processed)
test_selected = test_processed[selected_features]

feature_report.to_csv(FEATURE_REPORT_PATH, index=False)
print(f"Selected feature count: {len(selected_features)}")
display(feature_report.head(20))

## 8. Isolation Forest Training

In [ ]:
model = train_model(train_selected)
model

## 9. Anomaly Scoring

In [ ]:
test_predictions = predict_anomalies(model, test_selected)
test_scores = compute_anomaly_scores(model, test_selected)

record_ids = test_data["cell_id"] if "cell_id" in test_data.columns else test_data.index + 1
scores_df = scores_to_dataframe(test_scores, predictions=test_predictions, record_ids=record_ids)

display(scores_df.head())
display(evaluation_summary(scores_df))
display(to_json_records(scores_df.head(1)))

## 10. Save Outputs

In [ ]:
anomaly_scores_path = save_anomaly_scores(scores_df)
model_path = save_model(model, MODEL_PATH)
joblib.dump(preprocessing_pipeline, PREPROCESSING_PIPELINE_PATH)
feature_report.to_csv(FEATURE_REPORT_PATH, index=False)

print(f"Saved anomaly scores: {anomaly_scores_path}")
print(f"Saved Isolation Forest model: {model_path}")
print(f"Saved preprocessing pipeline: {PREPROCESSING_PIPELINE_PATH}")
print(f"Saved feature report: {FEATURE_REPORT_PATH}")